# Contour Implementation of 3-D Slab

In [ ]:
import os, sys
import numpy as np
import pyvista as pv
from pathlib import Path
from scipy.spatial import cKDTree

# Create a local directory to output
local_dir = Path("/home/lochy/ASPECT_PROJECT/HaMaGeoLib/dtemp/WBContour")

if not local_dir.exists():
    local_dir.mkdir()

# Utility functions

In [ ]:

def export_contours_to_vtu(depths, contours, filename="contours.vtk", *,
                           outer_radius=2890e3,
                           output_lines=True):
    """
    Export contour lines to a VTU file using z = outer_radius - depth.

    Parameters
    ----------
    depths : list[float]
        Depth values (meters), positive downward.
    contours : list[np.ndarray]
        Each entry is an (N_i, 2) array of (x, y) points.
    filename : str
        Output VTU file name.
    outer_radius : float, keyword-only
        Reference radius (meters). Default = 2890 km.
    """

    points = []
    depth_data = []
    z_data = []

    if output_lines: 
        lines = []

    point_offset = 0

    for d, curve in zip(depths, contours):
        z = outer_radius - d
        npts = curve.shape[0]

        for i in range(npts):
            x, y = curve[i]
            points.append([x, y, z])
            depth_data.append(d)
            z_data.append(z)

        # VTK polyline connectivity
        if output_lines: 
            line = [npts] + list(range(point_offset, point_offset + npts))
            lines.append(line)

        point_offset += npts

    points = np.array(points)
    
    if output_lines: 
        lines_flat = np.hstack(lines)

    mesh = pv.PolyData(points)
    
    if output_lines: 
        mesh.lines = lines_flat

    # attach fields
    mesh["depth"] = np.array(depth_data)
    mesh["z"] = np.array(z_data)

    mesh.save(filename)

    print(f"Saved to {filename}")



def contours_to_xyz_arrays(depths, contours, *, outer_radius=2890e3):
    """
    Convert contour representation to flattened X, Y, Z arrays.

    Parameters
    ----------
    depths : list[float]
        Depth values (meters), positive downward.
    contours : list[np.ndarray]
        Each entry is an (N_i, 2) array of (x, y).
    outer_radius : float, keyword-only
        Reference radius (meters). Default = 2890 km.

    Returns
    -------
    Xs, Ys, Zs : np.ndarray
        1D arrays of coordinates.
    """

    Xs = []
    Ys = []
    Zs = []

    for d, curve in zip(depths, contours):
        z = outer_radius - d

        # unpack curve
        xs = curve[:, 0]
        ys = curve[:, 1]

        # same z for all points in this contour
        zs = np.full_like(xs, z)

        Xs.append(xs)
        Ys.append(ys)
        Zs.append(zs)

    # concatenate into 1D arrays
    Xs = np.concatenate(Xs)
    Ys = np.concatenate(Ys)
    Zs = np.concatenate(Zs)

    return Xs, Ys, Zs



def write_structured_grid_to_vtu(Xg, Yg, Zg, filename="surface.vtu"):
    """
    Write a structured surface grid (Xg, Yg, Zg) to a VTU file.

    Parameters
    ----------
    Xg, Yg, Zg : np.ndarray
        2D arrays of shape (Ny, Nx)
    filename : str
        Output file name
    """

    Ny, Nx = Xg.shape

    # -----------------------------------
    # Flatten points
    # -----------------------------------
    points = np.column_stack((
        Xg.ravel(),
        Yg.ravel(),
        Zg.ravel()
    ))

    # -----------------------------------
    # Build quad connectivity
    # -----------------------------------
    cells = []
    celltypes = []

    for i in range(Ny - 1):
        for j in range(Nx - 1):
            # indices of quad corners
            p0 = i * Nx + j
            p1 = p0 + 1
            p2 = p0 + Nx + 1
            p3 = p0 + Nx

            # VTK quad: 4 points
            cells.extend([4, p0, p1, p2, p3])
            celltypes.append(pv.CellType.QUAD)

    cells = np.array(cells)
    celltypes = np.array(celltypes)

    # -----------------------------------
    # Create mesh
    # -----------------------------------
    mesh = pv.UnstructuredGrid(cells, celltypes, points)

    # optional: attach scalar field
    mesh["Z"] = Zg.ravel()

    mesh.save(filename)

    print(f"Saved structured surface to {filename}")



def build_surface_kdtree(depths, contours, *, outer_radius=2890e3):
    """
    Build KDTree from contour surface points.

    Returns
    -------
    tree : cKDTree
    points : (N, 3) array
    meta : list of (contour_id, local_index)
    """
    points = []
    meta = []

    for cid, (d, curve) in enumerate(zip(depths, contours)):
        z = outer_radius - d

        for i, (x, y) in enumerate(curve):
            points.append([x, y, z])
            meta.append((cid, i))

    points = np.array(points)
    tree = cKDTree(points)

    return tree, points, meta


def nearest_point_on_segment(p, a, b):
    ap = p - a
    ab = b - a
    t = np.dot(ap, ab) / np.dot(ab, ab)
    t = np.clip(t, 0.0, 1.0)
    return a + t * ab


def find_nearest_point_kdtree_with_segment(
    point, tree, points, meta, depths, contours, *,
    outer_radius=2890e3, k=5
):
    point = np.asarray(point)

    dists, idxs = tree.query(point, k=k)
    if k == 1:
        idxs = [idxs]

    min_dist = np.inf
    best_q = None
    best_a = None
    best_b = None
    best_cid = None
    best_j = None

    for idx in idxs:
        cid, i = meta[idx]
        curve = contours[cid]
        z = outer_radius - depths[cid]

        for j in [i - 1, i]:
            if j < 0 or j >= len(curve) - 1:
                continue

            a = np.array([curve[j][0], curve[j][1], z])
            b = np.array([curve[j+1][0], curve[j+1][1], z])

            # projection
            ap = point - a
            ab = b - a
            t = np.dot(ap, ab) / np.dot(ab, ab)
            t = np.clip(t, 0.0, 1.0)

            q = a + t * ab
            dist = np.linalg.norm(q - point)

            if dist < min_dist:
                min_dist = dist
                best_q = q
                best_a = a
                best_b = b
                best_j = j
                best_cid = cid

    return best_q, min_dist, [best_a, best_b, best_cid, best_j]



def export_point_connections_to_vtu(points, nearests, filename="connections.vtk"):
    """
    Export lines connecting points to their nearest surface points.

    Parameters
    ----------
    points : (N, 3) array-like
        Query points
    nearests : (N, 3) array-like
        Corresponding nearest surface points
    filename : str
        Output VTU file
    """

    points = np.asarray(points)
    nearests = np.asarray(nearests)

    assert points.shape == nearests.shape
    assert points.shape[1] == 3

    N = points.shape[0]

    # -----------------------------------
    # Combine all points
    # -----------------------------------
    all_points = np.vstack((points, nearests))

    # -----------------------------------
    # Build line connectivity
    # -----------------------------------
    lines = []

    for i in range(N):
        p_idx = i
        n_idx = i + N

        # each line: [2, point_id_1, point_id_2]
        lines.extend([2, p_idx, n_idx])

    lines = np.array(lines)

    # -----------------------------------
    # Create mesh
    # -----------------------------------
    mesh = pv.PolyData(all_points)
    mesh.lines = lines

    # -----------------------------------
    # Optional: add distance scalar
    # -----------------------------------
    distances = np.linalg.norm(points - nearests, axis=1)

    # duplicate for both endpoints
    mesh["distance"] = np.concatenate((distances, distances))

    mesh.save(filename)

    print(f"Saved connections to {filename}")



def nearest_segment_on_contour(point, contour, depth, *, outer_radius=2890e3):
    """
    Find nearest segment and point on a single contour.
    """
    z = outer_radius - depth
    p = np.asarray(point)

    min_dist = np.inf
    best_q = None
    best_a = None
    best_b = None

    for j in range(len(contour) - 1):
        a = np.array([contour[j][0], contour[j][1], z])
        b = np.array([contour[j+1][0], contour[j+1][1], z])

        # projection
        ap = p - a
        ab = b - a
        t = np.dot(ap, ab) / np.dot(ab, ab)
        t = np.clip(t, 0.0, 1.0)

        q = a + t * ab
        dist = np.linalg.norm(q - p)

        if dist < min_dist:
            min_dist = dist
            best_q = q
            best_a = a
            best_b = b
            best_j = j

    return best_q, best_a, best_b, min_dist, best_j



def build_local_patches(point, seg_info, contours, depths, *, outer_radius=2890e3):
    """
    Build local surface patches using known segment + adjacent contours.

    Parameters
    ----------
    point : (3,)
    seg_info : [a, b, cid]
        Output from KDTree stage
    contours, depths : lists
    """

    a0, b0, cid, a0_contour_index = seg_info
    patches = []
    patches_info = []

    # check neighbors only
    for cid2 in [cid - 1, cid + 1]:
        if cid2 < 0 or cid2 >= len(contours):
            continue

        # find nearest segment on neighbor contour
        q2, a2, b2, _, a2_contour_index = nearest_segment_on_contour(
            point,
            contours[cid2],
            depths[cid2],
            outer_radius=outer_radius
        )

        patches.append((a0, b0, a2, b2))

        # The patch info records the contour index and the point index on their contours
        # for each of the point. The order is (a, b, b2, a2), following either a clockwise
        # or an anti-clockwise order, depending on whether b2, a2 are on the previous or the
        # next contour to a, b.
        patch_info = (
        (cid, a0_contour_index),
        (cid, a0_contour_index+1),
        (cid2, a2_contour_index+1),
        (cid2, a2_contour_index),
        )
        patches_info.append(patch_info)


    return patches, patches_info



def refine_with_patches(point, patches, patches_info):
    """
    Refine nearest point using local surface patches.

    Parameters
    ----------
    point : (3,)
    patches : list of (a, b, a2, b2)

    Returns
    -------
    best_q : (3,)
    min_dist : float
    """

    point = np.asarray(point)

    best_q = None
    best_patch_index = None
    best_triangle_index = None
    best_triangle_bary = None
    min_dist = np.inf

    for i, patch in enumerate(patches):

        (a, b, a2, b2) = patch

        # split quad into two triangles
        triangles = [
            (a, b, a2),
            (b, b2, a2)
        ]

        for j, triangle in enumerate(triangles):

            (p0, p1, p2) = triangle

            q, dist, bary = closest_point_on_triangle(point, p0, p1, p2)

            if dist < min_dist:
                min_dist = dist
                best_q = q
                best_patch_index = i
                best_triangle_index = j
                best_triangle_bary = bary

    # We need to provide the contour number and index of the triangle points
    # as well as the triangle bary for later interpolation
    best_patch_info = patches_info[best_patch_index]
    best_triangle_info = (
        (best_patch_info[0], best_patch_info[1], best_patch_info[3])   # (a, b, a2)
        if best_triangle_index == 0
        else (best_patch_info[1], best_patch_info[2], best_patch_info[3]) # (b, b2, a2)
    )

    return best_q, min_dist, best_triangle_info, best_triangle_bary



def closest_point_on_triangle(p, a, b, c):
    """
    Compute the closest point on triangle (a, b, c) to point p.

    Returns
    -------
    q : (3,) ndarray
        Closest point
    dist : float
        Distance to p
    bary : (alpha, beta, gamma)
        Barycentric coordinates (sum to 1)
    """

    p = np.asarray(p)
    a = np.asarray(a)
    b = np.asarray(b)
    c = np.asarray(c)

    ab = b - a
    ac = c - a
    ap = p - a

    # Vertex A
    d1 = np.dot(ab, ap)
    d2 = np.dot(ac, ap)
    if d1 <= 0.0 and d2 <= 0.0:
        q = a
        bary = (1.0, 0.0, 0.0)

    else:
        # Vertex B
        bp = p - b
        d3 = np.dot(ab, bp)
        d4 = np.dot(ac, bp)
        if d3 >= 0.0 and d4 <= d3:
            q = b
            bary = (0.0, 1.0, 0.0)

        else:
            # Edge AB
            vc = d1 * d4 - d3 * d2
            if vc <= 0.0 and d1 >= 0.0 and d3 <= 0.0:
                v = d1 / (d1 - d3)
                q = a + v * ab
                bary = (1 - v, v, 0.0)

            else:
                # Vertex C
                cp = p - c
                d5 = np.dot(ab, cp)
                d6 = np.dot(ac, cp)
                if d6 >= 0.0 and d5 <= d6:
                    q = c
                    bary = (0.0, 0.0, 1.0)

                else:
                    # Edge AC
                    vb = d5 * d2 - d1 * d6
                    if vb <= 0.0 and d2 >= 0.0 and d6 <= 0.0:
                        w = d2 / (d2 - d6)
                        q = a + w * ac
                        bary = (1 - w, 0.0, w)

                    else:
                        # Edge BC
                        va = d3 * d6 - d5 * d4
                        if va <= 0.0 and (d4 - d3) >= 0.0 and (d5 - d6) >= 0.0:
                            w = (d4 - d3) / ((d4 - d3) + (d5 - d6))
                            q = b + w * (c - b)
                            bary = (0.0, 1 - w, w)

                        else:
                            # Inside face
                            denom = 1.0 / (va + vb + vc)
                            v = vb * denom
                            w = vc * denom
                            u = 1.0 - v - w
                            q = a + v * ab + w * ac
                            bary = (u, v, w)

    dist = np.linalg.norm(q - p)

    return q, dist, bary



def ray_segment_intersection(T, n, A, B):
    """
    Solve T + s*n = A + u*(B-A)

    Returns
    -------
    q : (2,) array
    s : float  (distance along ray)
    u : float  (fraction along segment [0,1])
    or None if no valid intersection
    """

    v = B - A

    M = np.column_stack((n, -v))  # 2x2
    rhs = A - T

    det = np.linalg.det(M)
    if abs(det) < 1e-10:
        return None

    sol = np.linalg.solve(M, rhs)
    s, u = sol

    if 0 <= u <= 1:
        q = T + s * n
        return q, s, u

    return None



def compute_tangent_normal(contour):
    """
    Compute unit tangent and normal vectors along a 2D contour.

    Parameters
    ----------
    contour : (N, 2) array
        Sequence of (x, y) points defining the curve

    Returns
    -------
    tangents : (N, 2) array
    normals  : (N, 2) array
    """

    contour = np.asarray(contour)
    N = len(contour)

    tangents = np.zeros_like(contour)
    normals = np.zeros_like(contour)

    for i in range(N):
        if i == 0:
            # forward difference
            t = contour[i+1] - contour[i]
        elif i == N - 1:
            # backward difference
            t = contour[i] - contour[i-1]
        else:
            # central difference (better)
            t = contour[i+1] - contour[i-1]

        # normalize tangent
        norm = np.linalg.norm(t)
        if norm == 0:
            continue
        t = t / norm

        # rotate 90° → normal
        n = np.array([-t[1], t[0]])

        tangents[i] = t
        normals[i] = n

    return tangents, normals



def find_ray_contour_intersection(T, n, contour):
    """
    Returns
    -------
    mapped_point : (2,)
    ray_distance : float
    fractional_index : float
    """

    mapped_point = None
    ray_distance = np.inf
    fractional_index = None

    for j in range(len(contour) - 1):
        A = contour[j]
        B = contour[j + 1]

        result = ray_segment_intersection(T, n, A, B)

        if result is not None:
            q, s, u = result

            if s < ray_distance:
                ray_distance = s
                mapped_point = q
                fractional_index = j + u

    return mapped_point, ray_distance, fractional_index



def map_trench_to_all_contours(contours):
    """
    Map trench (contours[0]) to all other contours.

    Parameters
    ----------
    contours : list of (Ni, 2) arrays

    Returns
    -------
    mapped_points_list : list of arrays (N_trench, 2)
    ray_distances_list : list of arrays (N_trench,)
    fractional_indices_list : list of arrays (N_trench,)
    """

    trench = contours[0]
    tangents, normals = compute_tangent_normal(trench)

    mapped_points_list = []
    ray_distances_list = []
    fractional_indices_list = []

    # loop over all other contours
    for k in range(1, len(contours)):
        contour_B = contours[k]

        mapped_points = []
        ray_distances = []
        fractional_indices = []

        for i, T in enumerate(trench):
            n = normals[i]

            q, s, frac_idx = find_ray_contour_intersection(T, n, contour_B)

            if q is None:
                mapped_points.append([np.nan, np.nan])
                ray_distances.append(np.nan)
                fractional_indices.append(np.nan)
            else:
                
                # Here we need to make sure that the mapping points don't 
                # contain any duplicated, such that the frac_idx must
                # keep increasing
                is_duplicate = False
                for pre_frac_idx in fractional_indices:
                    if frac_idx < pre_frac_idx:
                        is_duplicate = True

                if is_duplicate:
                    mapped_points.append([np.nan, np.nan])
                    ray_distances.append(np.nan)
                    fractional_indices.append(np.nan)
                
                else:
                    mapped_points.append(q)
                    ray_distances.append(s)
                    fractional_indices.append(frac_idx)

        mapped_points_list.append(np.array(mapped_points))
        ray_distances_list.append(np.array(ray_distances))
        fractional_indices_list.append(np.array(fractional_indices))

    return mapped_points_list, ray_distances_list, fractional_indices_list



def export_mapped_points_to_vtu(
    mapped_points_list,
    ray_distances_list,
    fractional_indices_list,
    depths,
    filename="mapped_points.vtk",
    *,
    outer_radius=2890e3
):
    """
    Export mapped points with metadata to VTU.

    Parameters
    ----------
    mapped_points_list : list of (Nt, 2)
    ray_distances_list : list of (Nt,)
    fractional_indices_list : list of (Nt,)
    depths : list of float
        depths corresponding to contours[1:]
    """

    points = []
    trench_indices = []
    fractional_indices = []
    ray_distances = []
    contour_ids = []

    Nt = mapped_points_list[0].shape[0]

    for k in range(len(mapped_points_list)):
        mapped_points = mapped_points_list[k]
        ray_dist = ray_distances_list[k]
        frac_idx = fractional_indices_list[k]

        z = outer_radius - depths[k+1]  # skip trench depth

        for i in range(Nt):
            x, y = mapped_points[i]

            if np.isnan(x) or np.isnan(y):
                continue  # skip invalid points

            points.append([x, y, z])

            trench_indices.append(i)
            fractional_indices.append(frac_idx[i])
            ray_distances.append(ray_dist[i])
            contour_ids.append(k + 1)

    # convert to arrays
    points = np.array(points)

    mesh = pv.PolyData(points)

    # attach fields
    mesh["trench_index"] = np.array(trench_indices)
    mesh["fractional_index"] = np.array(fractional_indices)
    mesh["ray_distance"] = np.array(ray_distances)
    mesh["contour_id"] = np.array(contour_ids)

    mesh.save(filename)

    print(f"Saved mapped points to {filename}")




def evaluate_contour_at_fractional_index(contour, fractional_index):
    """
    Compute point on contour given fractional_index.

    Parameters
    ----------
    contour : (N, 2) array
    fractional_index : float

    Returns
    -------
    point : (2,) array

    Raises
    ------
    ValueError
        If fractional_index is NaN or out of valid range
    """

    if np.isnan(fractional_index):
        raise ValueError("fractional_index is NaN")

    N = len(contour)

    j = int(np.floor(fractional_index))
    u = fractional_index - j

    # todo_contour
    # Points are allowed to sit outside of the original contours
    # and this is done by extending the first and the last line segments
    if j < 0 :
        A = contour[0]
        B = contour[1]

        return j * (B - A) + A

    elif j >= N - 1:
        A = contour[N-2]
        B = contour[N-1]
        
        return (j - N + 1) * (B - A) + B

    A = contour[j]
    B = contour[j + 1]

    return (1 - u) * A + u * B



def map_contour_points_to_trench(trench, contour, fractional_indices):
    """
    Map contour points to trench using monotonic fractional_indices.
    Out-of-range values → NaN.
    """


    Nt = len(trench)

    valid = ~np.isnan(fractional_indices)

    s_vals = fractional_indices[valid]
    i_vals = np.arange(Nt)[valid]

    s_min = s_vals[0]
    contour_mapped_point_start = evaluate_contour_at_fractional_index(contour, s_min)
    s_max = s_vals[-1]
    contour_mapped_point_end = evaluate_contour_at_fractional_index(contour, s_max)

    trench_mapped = []
    trench_indices_mapped = []

    for j in range(len(contour)):
        s = j

        # range check, if the index is out of the fractional index
        # migrate from the starting / end point of the trench,
        # In relatiion, give negative indices or indices larger than
        # Nt - 1
        if s < s_min:
            if np.isnan(fractional_indices[0]):
                raise ValueError("With out of range fractional index (s < s_min), \
                                trench starting point must have mapped poiont on the local contour")
            
            x_migration, y_migration = contour[j] - contour_mapped_point_start
            
            trench_mapped.append([trench[0][0] + x_migration, trench[0][1] + y_migration])

            extension = np.linalg.norm([x_migration, y_migration])
            first_segment = np.linalg.norm(trench[1]-trench[0])

            trench_indices_mapped.append(-extension/first_segment)
            continue

        elif s > s_max:
            if np.isnan(fractional_indices[-1]):
                raise ValueError("With out of range fractional index (s > s_max), \
                                trench ending point must have mapped poiont on the local contour")
            
            x_migration, y_migration = contour[j] - contour_mapped_point_end
            

            trench_mapped.append([trench[-1][0] + x_migration, trench[-1][1] + y_migration])
            
            extension = np.linalg.norm([x_migration, y_migration])
            last_segment = np.linalg.norm(trench[-1]-trench[-2])
            
            trench_indices_mapped.append(Nt - 1 + extension/last_segment)
            continue

        # interpolate index
        i = np.interp(s, s_vals, i_vals)

        i0 = int(np.floor(i))
        i1 = min(i0 + 1, Nt - 1)

        u = i - i0

        pt = (1 - u) * trench[i0] + u * trench[i1]

        # todo_contour
        trench_mapped.append(pt)
        trench_indices_mapped.append(i)

    return np.array(trench_mapped), np.array(trench_indices_mapped)



def build_trench_mapped_list(contours, fractional_indices_list):
    """
    Build trench-mapped points for all contours using existing mapping function.

    Parameters
    ----------
    contours : list of (Ni, 2)
    fractional_indices_list : list of (Nt,)

    Returns
    -------
    trench_mapped_list : list of (Ni, 2)
    """

    trench = contours[0]

    trench_mapped_list = []
    trench_mapped_index_list = []

    for k in range(1, len(contours)):
        contour = contours[k]
        fractional_indices = fractional_indices_list[k-1]

        trench_mapped, trench_indices_mapped = map_contour_points_to_trench(
            trench,
            contour,
            fractional_indices
        )

        trench_mapped_list.append(trench_mapped)
        trench_mapped_index_list.append(trench_indices_mapped)

    return trench_mapped_list, trench_mapped_index_list



def export_contour_trench_connections_3d(
    contour,
    trench_mapped,
    depth_contour,
    depth_trench=0.0,
    *,
    outer_radius=2890e3,
    filename="contour_trench_connections.vtk"
):
    """
    Export 3D line connections between contour and mapped trench points.

    Parameters
    ----------
    contour : (N, 2)
    trench_mapped : (N, 2)
    depth_contour : float
    depth_trench : float (default = 0)
    outer_radius : float
    filename : str
    """

    contour = np.asarray(contour)
    trench_mapped = np.asarray(trench_mapped)

    N = len(contour)

    points = []
    lines = []
    distances = []
    ray_distances = []

    z_c = outer_radius - depth_contour
    z_t = outer_radius - depth_trench

    for i in range(N):
        c = contour[i]
        t = trench_mapped[i]

        # skip invalid mappings
        if np.isnan(c[0]) or np.isnan(t[0]):
            continue

        idx_c = len(points)
        points.append([c[0], c[1], z_c])

        idx_t = len(points)
        points.append([t[0], t[1], z_t])

        lines.extend([2, idx_c, idx_t])

        # 3D distance
        d = np.linalg.norm(
            np.array([c[0], c[1], z_c]) -
            np.array([t[0], t[1], z_t])
        )

        distances.extend([d, d])

        # vertical ray distance (optional)
        ray_distances.extend([depth_contour, depth_contour])

    mesh = pv.PolyData(np.array(points))
    mesh.lines = np.array(lines)

    mesh["distance_3d"] = np.array(distances)
    mesh["depth"] = np.array(ray_distances)

    mesh.save(filename)

    print(f"Saved {filename}")




# Synthetic Test Contours

For the first contour, a sin function features the shape of the trench.

In [ ]:
contour_type = "sin_half_wavelength"

## Contour with a trench of sin function

### Trench of a sin-function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# x range
x = np.linspace(1000e3, 2000e3, 500)

# normalize to 0-1
xn = (x - 1000e3) / (2000e3 - 1000e3)

# half wavelength over the whole x range
y = 3000e3 + 400e3 * np.sin(np.pi * xn)

plt.figure(figsize=(6, 5))
plt.plot(x/1e3, y/1e3)
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.title("Synthetic Trench (Sin Function)")

fig_path = local_dir/"trench_initial.png"
plt.savefig(fig_path)
print("saved figure " + str(fig_path))

plt.show()

### Synthetic contours at different depths

These are created by migrating the trench curve to negative y direction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# -----------------------------------
# Variables
# -----------------------------------
outer_radius = 2890e3
Nxy = 20 # number of points on a contour
Dz = 100e3 # depth interval of controus

# -----------------------------------
# Base curve (surface)
# -----------------------------------
x = np.linspace(1000e3, 2000e3, Nxy)
xn = (x - 1000e3) / (2000e3 - 1000e3)

y0 = 3000e3 + 400e3 * np.sin(np.pi * xn)

# -----------------------------------
# Depth levels
# -----------------------------------
depths = list(np.arange(0, 600e3, Dz))  # [0, 100km, 200km, ...]

# -----------------------------------
# Generate contours (parallel list)
# -----------------------------------
contours = []

for d in depths:
    y = y0 - d
    curve = np.column_stack((x, y))
    contours.append(curve)

# -----------------------------------
# Plot
# -----------------------------------
plt.figure(figsize=(7, 6))

for d, curve in zip(depths, contours):
    plt.plot(curve[:, 0], curve[:, 1], label=f"{int(d/1e3)} km")

plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("Synthetic depth contours")
plt.legend()
plt.gca().invert_yaxis()

fig_path = local_dir/"contour_initial.png"
plt.savefig(fig_path)
print("saved figure " + str(fig_path))

plt.show()

In [ ]:
export_contours_to_vtu(depths, contours, filename=local_dir/"contours.vtk",
                       output_lines=False)

# Test query points

In [ ]:
ps = [
np.array([1950e3, 2566e3, 2390e3]),
np.array([1100e3, 2630e3, 2590e3]),
np.array([1400e3, 2900e3, 2700e3]),
np.array([1450e3, 2950e3, 2650e3]),
np.array([1500e3, 3000e3, 2600e3]),
np.array([1550e3, 3050e3, 2550e3]),
np.array([1600e3, 3100e3, 2500e3]),
]

# KNN interpolation for nearest point

The cost for the algorithm

O(NlogN)+O(Q(logN+k))

N - total points
Q - query points
k - k near-neighbors

In [ ]:
from copy import deepcopy


# build once
tree, points, meta = build_surface_kdtree(depths, contours)

nearests = []
min_dists = []
triangles_info = []
triangles_bary = []
for i_p, p in enumerate(ps):
    # query many times
    nearest, dist, seg_info = find_nearest_point_kdtree_with_segment(
        p, tree, points, meta, depths, contours, k=5
    )

    # get two patches that have the line segment we get in the last step
    # in order to map point back to the trench in the latter step, we also
    # need to collest the index of the contour and the points, as well as
    # the triangle bary
    patches, patches_info = build_local_patches(p, seg_info, contours, depths)

    nearest, min_dist, best_triangle_info, best_triangle_bary = refine_with_patches(p, patches, patches_info)

    nearests.append(deepcopy(nearest))
    min_dists.append(min_dist)
    triangles_info.append(best_triangle_info)
    triangles_bary.append(best_triangle_bary)


export_point_connections_to_vtu(ps, nearests, local_dir/"nearest_connections.vtk")

# Ray trace between trench points and contour points

## Ray-trace trench points to contour locations

In [ ]:
mapped_points_list, ray_distances_list, fractional_indices_list = map_trench_to_all_contours(contours)

export_mapped_points_to_vtu(
    mapped_points_list,
    ray_distances_list,
    fractional_indices_list,
    depths,
    local_dir/"mapped_contour_points.vtk"
)

## Trace back from the points on each of the contour points backwards to positions on the trench.

In [ ]:
# trace back from contours to trench

trench_mapped_list, trench_index_mapped_list = build_trench_mapped_list(contours, fractional_indices_list)


k = len(contours) - 1  # contour index

export_contour_trench_connections_3d(
    contours[k],
    trench_mapped_list[k-1],
    depth_contour=depths[k],
    depth_trench=depths[0],  # usually 0
    filename=local_dir/f"connections_3d_{k}.vtk"
)

# Mapping point on the trench

In [ ]:
trench_positions = []
trench_positions_3d = []

# todo_contour
print("trench_index_mapped_list[-1]: ")
print(trench_index_mapped_list[-1])

for i_p, p in enumerate(ps):

    # Get the triange and bary
    triangle_info = triangles_info[i_p]
    triangle_bary = triangles_bary[i_p]

    trench_fractional_index = 0
    for j_p1, (cid_p1, index_p1) in enumerate(triangle_info):

        trench_fractional_index += triangle_bary[j_p1]* trench_index_mapped_list[cid_p1-1][index_p1]
        
        # todo_contour
        if i_p == 0:
            print("cid_p1: ", cid_p1)
            print("index_p1: ", index_p1)
            print("trench_index_mapped_list[cid_p1-1][index_p1]: ", trench_index_mapped_list[cid_p1-1][index_p1])
    
    trench_position = evaluate_contour_at_fractional_index(contours[0], trench_fractional_index)

    trench_positions.append(trench_position)
    trench_positions_3d.append(np.array([trench_position[0], trench_position[1], outer_radius]))


# todo_contour
print("trench_positions: ")
print(trench_positions)
    
export_point_connections_to_vtu(nearests, trench_positions_3d, local_dir/"trench_connections.vtk")